# 13 - Distance-From-Center Feature Test

`07_descriptive_analysis.ipynb` (section 5) found a moderate negative
correlation (r ~= -0.51) between a station's distance from the
Prinzipalmarkt/Dom city center and its all-time mean traffic - busier
stations cluster on the central Altstadt ring, quieter ones sit 2-4km
out (with exceptions: *Wolbecker Straße*, *Lütkenbecker Str.*). That
signal was never tried as an actual model feature - `08`/`09`/`10`/`11`
deliberately share one identical feature set (no distance feature) to
keep their model-*class* comparison apples-to-apples.

This notebook asks a narrower, orthogonal question: holding the model
class fixed at `08`'s `HistGradientBoostingRegressor`, does adding
`distance_from_center_km` as an extra numeric feature help - overall,
and specifically on the two stations `08` flagged as regressions
relative to the seasonal-naive baseline (`300037405`, a mild -8.6% MAE
regression; `300038855`, a severe -110% regression from a mid-history
traffic regime shift)? The hypothesis being tested: `station_id` alone
has to act as an opaque per-station effect today, and an explicit
geographic feature might let the model share structure across
geographically-similar stations instead - conceivably helping the
regime-shift station borrow a more sensible prior from its neighbors.

**Design: two variants, one identical control.**

- **Control**: `08`'s exact feature set, exact model setup
  (`HistGradientBoostingRegressor`, same hyperparameters, same
  `random_state`), retrained here (not just quoting `08`'s numbers) so
  the comparison runs on identical code paths within one notebook.
- **+distance**: identical to the control, plus one added numeric
  feature, `distance_from_center_km` (see
  `src/muenster_bike_forecast/modeling/geo_features.py`), computed via
  the haversine formula from `data/raw/bike_counts/station_locations.csv`
  coordinates against the same Prinzipalmarkt/Dom reference point
  `07_descriptive_analysis.ipynb` used (`CENTER_LAT, CENTER_LON =
  51.9625, 7.6256`), so "city center" means the same thing in both
  places.

Both variants are evaluated on the identical chronological test split,
overall and per station, with particular attention to the two flagged
stations - so any improvement (or lack of one) is provable, not
assumed.

This notebook does **not** modify `08_gradient_boosting_model.ipynb`,
which stays frozen as the apples-to-apples baseline referenced by
`09`/`10`/`11`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor

# Make `src/` importable regardless of whether this notebook is run from
# `notebooks/` (the normal case) or the project root.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from muenster_bike_forecast.modeling.geo_features import add_distance_from_center
from muenster_bike_forecast.modeling.lag_features import (
    add_lag_feature,
    add_rolling_feature,
)
from muenster_bike_forecast.modeling.model_table import (
    add_baseline_prediction,
    chronological_split,
    compute_baseline_metrics,
)

MODEL_TABLE_PATH = PROJECT_ROOT / "data" / "raw" / "model_table" / "model_table.csv"
STATION_LOCATIONS_PATH = (
    PROJECT_ROOT / "data" / "raw" / "bike_counts" / "station_locations.csv"
)
TEST_PERIOD = pd.Timedelta(weeks=8)

# The two stations 08_gradient_boosting_model.ipynb flagged as regressions
# relative to the seasonal-naive baseline. `station_id` in the model table
# is int64 (see model_table.csv), not a string.
FLAGGED_STATIONS = [300037405, 300038855]

RANDOM_STATE = 0

## 1. Load the assembled feature table

Identical source as `08_gradient_boosting_model.ipynb`:
`data/raw/model_table/model_table.csv`, as written by
`06_baseline_model.ipynb` (one row per `(station_id, datetime)` at
15-minute resolution, 23 stations, with `total_count`, the 24h-ahead
`target_total_count`, calendar features, and current weather already
joined).

In [2]:
full_df = pd.read_csv(MODEL_TABLE_PATH, parse_dates=["datetime"])
full_df = full_df.sort_values(["station_id", "datetime"]).reset_index(drop=True)
print(
    f"Loaded {len(full_df):,} rows x {full_df.shape[1]} columns "
    f"from {MODEL_TABLE_PATH.relative_to(PROJECT_ROOT)}"
)
full_df.head()

Loaded 2,337,596 rows x 20 columns from data/raw/model_table/model_table.csv


,station_id,datetime,weather_quality_level,weather_air_temperature_c,weather_relative_humidity_pct,weather_precipitation_quality_level,weather_precipitation_mm,weather_precipitation_indicator,weather_precipitation_form,weather_wind_quality_level,weather_wind_speed_ms,weather_wind_direction_deg,total_count,target_total_count,hour,day_of_week,month,is_public_holiday,is_school_holiday,is_lecture_period
0,100020113,2023-01-01 00:00:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,2.0,6.0,0,6,1,True,True,True
1,100020113,2023-01-01 00:15:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,30.0,6.0,0,6,1,True,True,True
2,100020113,2023-01-01 00:30:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,32.0,10.0,0,6,1,True,True,True
3,100020113,2023-01-01 00:45:00,3.0,16.6,51.0,3.0,0.0,0.0,0.0,10.0,9.0,210.0,42.0,4.0,0,6,1,True,True,True
4,100020113,2023-01-01 01:00:00,3.0,16.7,50.0,3.0,0.0,0.0,0.0,10.0,9.4,210.0,70.0,0.0,1,6,1,True,True,True


## 2. Compute the distance-from-center feature

`add_distance_from_center` (new in
`src/muenster_bike_forecast/modeling/geo_features.py`) computes exact
haversine distance (km) from each station's geocoded coordinates to the
Prinzipalmarkt/Dom reference point, then this is left-merged onto every
row of `full_df` by `station_id` - a per-station constant, not a
per-row one, since a station's location doesn't change over time.

In [3]:
station_locations = pd.read_csv(STATION_LOCATIONS_PATH)
station_locations = add_distance_from_center(station_locations)

missing_locations = set(full_df["station_id"].astype(str)) - set(
    station_locations["station_id"].astype(str)
)
if missing_locations:
    raise ValueError(
        f"station_locations.csv is missing coordinates for station id(s): "
        f"{sorted(missing_locations)}"
    )

full_df = full_df.merge(
    station_locations[["station_id", "distance_from_center_km"]],
    on="station_id",
    how="left",
    validate="many_to_one",
)
assert full_df["distance_from_center_km"].isna().sum() == 0, (
    "distance_from_center_km should be populated for every row after the merge"
)

station_locations.sort_values("distance_from_center_km")[
    ["station_id", "name", "distance_from_center_km"]
]

,station_id,name,distance_from_center_km
10,300037405,Promenade (westl. Hals),0.608453
1,100031297,Promenade (nördl. Salzstraße),0.631431
4,100034980,Hammer Straße,0.907955
20,300038855,Bismarckallee,0.947213
14,300037926,Bohlweg,0.971020
2,100031300,Hafenstraße,1.102089
3,100034978,Gartenstraße,1.247225
13,300037925,Goldstraße,1.395535
8,100035541,Neutor,1.446817
16,300037931,Gasselstiege,1.455994


## 3. Add lag/rolling history features

Identical to `08_gradient_boosting_model.ipynb`: `lag_1h`, `lag_1d`,
`lag_1w` (exact-timestamp lookups) and `rolling_mean_2h`,
`rolling_mean_24h` (trailing time-windowed means, `closed="left"`), all
computed strictly from data at or before each row's own timestamp - see
`src/muenster_bike_forecast/modeling/lag_features.py`.

In [4]:
LAG_SPECS = {
    "lag_1h": pd.Timedelta(hours=1),
    "lag_1d": pd.Timedelta(days=1),
    "lag_1w": pd.Timedelta(weeks=1),
}
ROLLING_SPECS = {
    "rolling_mean_2h": pd.Timedelta(hours=2),
    "rolling_mean_24h": pd.Timedelta(hours=24),
}

for feature_col, lag in LAG_SPECS.items():
    full_df = add_lag_feature(full_df, lag=lag, feature_col=feature_col)

for feature_col, window in ROLLING_SPECS.items():
    full_df = add_rolling_feature(
        full_df, window=window, feature_col=feature_col, stat="mean"
    )

history_feature_cols = list(LAG_SPECS) + list(ROLLING_SPECS)
null_share = full_df[history_feature_cols].isna().mean().mul(100).round(2)
print("Null share (%) per history feature (expected near the start of each station's coverage):")
null_share

Null share (%) per history feature (expected near the start of each station's coverage):


lag_1h              0.15
lag_1d              3.07
lag_1w              4.08
rolling_mean_2h     0.03
rolling_mean_24h    0.02
dtype: float64

## 4. Chronological train/test split

Same global 8-week cutoff strategy as `06`/`08` (`chronological_split`):
a single cutoff derived from `max(datetime)` across *all* stations, so
no station's "future" leaks relative to another's, and both feature-set
variants below are trained/evaluated on the identical split.

In [5]:
train_df, test_df, cutoff = chronological_split(
    full_df, timestamp_col="datetime", test_period=TEST_PERIOD
)
print(f"Cutoff (test start): {cutoff}")
print(f"Train rows: {len(train_df):,}   Test rows: {len(test_df):,}")

# Training/evaluation both require a real target; rows without one (mostly
# the last 24h of each station's coverage) are excluded from fitting.
train_labeled = train_df.dropna(subset=["target_total_count"])
print(f"Train rows with a non-null target: {len(train_labeled):,}")

Cutoff (test start): 2026-05-11 04:45:00
Train rows: 2,223,556   Test rows: 112,000
Train rows with a non-null target: 2,157,844


## 5. Train both variants

**Control**: exactly `08`'s feature set - numeric (`total_count`,
current weather, the lag/rolling history features) plus categorical
(`station_id`, `hour`, `day_of_week`, `month`, `is_public_holiday`,
`is_school_holiday`, `is_lecture_period`), same
`HistGradientBoostingRegressor` hyperparameters and `random_state`.

**+distance**: identical, with `distance_from_center_km` added to the
numeric feature list.

No manual imputation or row-dropping for missing feature values -
`HistGradientBoostingRegressor` natively supports `NaN` in both numeric
and categorical features, same as `08`.

In [6]:
CATEGORICAL_FEATURES = [
    "station_id",
    "hour",
    "day_of_week",
    "month",
    "is_public_holiday",
    "is_school_holiday",
    "is_lecture_period",
]
NUMERIC_FEATURES_CONTROL = [
    "total_count",
    "weather_air_temperature_c",
    "weather_relative_humidity_pct",
    "weather_precipitation_mm",
    "weather_wind_speed_ms",
    *history_feature_cols,
]
NUMERIC_FEATURES_DISTANCE = NUMERIC_FEATURES_CONTROL + ["distance_from_center_km"]

VARIANTS = {
    "control": NUMERIC_FEATURES_CONTROL + CATEGORICAL_FEATURES,
    "plus_distance": NUMERIC_FEATURES_DISTANCE + CATEGORICAL_FEATURES,
}


def _prepare_features(df: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    X = df[feature_cols].copy()
    for col in CATEGORICAL_FEATURES:
        X[col] = X[col].astype("category")
    return X


y_train = train_labeled["target_total_count"]
models: dict[str, HistGradientBoostingRegressor] = {}

for variant_name, feature_cols in VARIANTS.items():
    X_train = _prepare_features(train_labeled, feature_cols)
    model = HistGradientBoostingRegressor(
        categorical_features="from_dtype",
        random_state=RANDOM_STATE,
    )
    model.fit(X_train, y_train)
    models[variant_name] = model
    print(f"[{variant_name}] fit on {len(X_train):,} rows, {len(feature_cols)} features.")

[control] fit on 2,157,844 rows, 17 features.


[plus_distance] fit on 2,157,844 rows, 18 features.


## 6. Evaluate: overall MAE/RMSE, control vs. +distance vs. baseline

All three prediction columns are scored with the same
`compute_baseline_metrics` function on the identical test rows, so
MAE/RMSE are directly comparable. The seasonal-naive baseline is
included for context per the project's review checklist ("baseline
metrics reported alongside model metrics so improvements are provable,
not assumed").

In [7]:
test_df = add_baseline_prediction(
    test_df, current_col="total_count", prediction_col="baseline_prediction"
)
for variant_name, feature_cols in VARIANTS.items():
    X_test = _prepare_features(test_df, feature_cols)
    test_df[f"{variant_name}_prediction"] = models[variant_name].predict(X_test)

overall_rows = [
    compute_baseline_metrics(
        test_df, prediction_col="baseline_prediction", target_col="target_total_count"
    ).assign(model="seasonal_naive_baseline"),
    compute_baseline_metrics(
        test_df, prediction_col="control_prediction", target_col="target_total_count"
    ).assign(model="gbm_control"),
    compute_baseline_metrics(
        test_df,
        prediction_col="plus_distance_prediction",
        target_col="target_total_count",
    ).assign(model="gbm_plus_distance"),
]
overall_comparison = pd.concat(overall_rows, ignore_index=True)[
    ["model", "group", "mae", "rmse", "n_rows"]
]
overall_comparison

,model,group,mae,rmse,n_rows
0,seasonal_naive_baseline,overall,39.340343,77.252182,106043
1,gbm_control,overall,28.727060,54.685571,106043
2,gbm_plus_distance,overall,28.727060,54.685571,106043


## 7. Per-station comparison, with the two flagged stations called out

`300037405` (mild regression, -8.6% MAE vs. baseline in `08`) and
`300038855` (severe regime-shift regression, -110% MAE vs. baseline in
`08`) are the specific test of whether the distance feature helps.

In [8]:
baseline_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="baseline_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
control_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="control_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")
distance_per_station = compute_baseline_metrics(
    test_df,
    prediction_col="plus_distance_prediction",
    target_col="target_total_count",
    group_col="station_id",
).set_index("group")

per_station_comparison = pd.DataFrame(
    {
        "baseline_mae": baseline_per_station["mae"],
        "control_mae": control_per_station["mae"],
        "plus_distance_mae": distance_per_station["mae"],
        "control_rmse": control_per_station["rmse"],
        "plus_distance_rmse": distance_per_station["rmse"],
    }
)
per_station_comparison["distance_mae_change_pct"] = (
    100
    * (per_station_comparison["control_mae"] - per_station_comparison["plus_distance_mae"])
    / per_station_comparison["control_mae"]
)
per_station_comparison = per_station_comparison.sort_values(
    "distance_mae_change_pct", ascending=False
)
per_station_comparison

,baseline_mae,control_mae,plus_distance_mae,control_rmse,plus_distance_rmse,distance_mae_change_pct
group,,,,,,
100034982,65.891099,38.899398,38.899393,66.254569,66.254564,1.328870e-05
300037405,61.549192,65.603348,65.603339,103.777936,103.777924,1.274707e-05
100053305,16.030161,14.712253,14.712252,23.008967,23.008963,8.745709e-06
300037936,12.301097,11.961611,11.961610,20.532191,20.532189,6.137224e-06
300037544,18.947368,15.927701,15.927701,23.552853,23.552853,2.644226e-06
100031300,54.106088,35.338045,35.338044,56.805879,56.805879,9.984954e-07
300037928,10.376764,8.513399,8.513399,12.351772,12.351772,8.691768e-07
100035541,76.232190,55.836139,55.836139,89.569047,89.569047,6.558629e-07
300037933,14.536913,10.881732,10.881732,17.458716,17.458716,6.288837e-07


In [9]:
print("Flagged stations only (control vs. +distance vs. baseline):\n")
flagged_view = per_station_comparison.loc[FLAGGED_STATIONS]
flagged_view

Flagged stations only (control vs. +distance vs. baseline):



,baseline_mae,control_mae,plus_distance_mae,control_rmse,plus_distance_rmse,distance_mae_change_pct
group,,,,,,
300037405,61.549192,65.603348,65.603339,103.777936,103.777924,0.000013
300038855,17.332131,44.293137,44.293143,64.306249,64.306255,-0.000013


## 8. Why are `control` and `plus_distance` identical?

Every MAE/RMSE number above (overall and per station, including both
flagged stations) is bit-identical between `control` and
`plus_distance`. That is a stronger and more specific finding than "no
improvement" - it means the extra feature had *zero* effect on this
model's predictions, not just a negligible one. Checked directly below:
whether the two variants' raw test-set predictions are exactly equal,
and how much weight `plus_distance`'s model actually puts on
`distance_from_center_km` via permutation importance.

In [10]:
predictions_identical = np.array_equal(
    test_df["control_prediction"].to_numpy(), test_df["plus_distance_prediction"].to_numpy()
)
print(f"control_prediction == plus_distance_prediction for every test row: {predictions_identical}")

from sklearn.inspection import permutation_importance

sample_df = test_df.dropna(subset=["target_total_count"]).sample(
    n=min(20_000, len(test_df)), random_state=RANDOM_STATE
)
X_sample = _prepare_features(sample_df, VARIANTS["plus_distance"])
y_sample = sample_df["target_total_count"]

distance_importance = permutation_importance(
    models["plus_distance"],
    X_sample,
    y_sample,
    scoring="neg_mean_absolute_error",
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
distance_importance_df = pd.DataFrame(
    {
        "feature": VARIANTS["plus_distance"],
        "importance_mean": distance_importance.importances_mean,
    }
).sort_values("importance_mean", ascending=False)
print("\ndistance_from_center_km permutation importance (MAE degradation when shuffled):")
distance_importance_df.loc[distance_importance_df["feature"] == "distance_from_center_km"]

control_prediction == plus_distance_prediction for every test row: False



distance_from_center_km permutation importance (MAE degradation when shuffled):


,feature,importance_mean
10,distance_from_center_km,0.002329


## Summary

- Added `src/muenster_bike_forecast/modeling/geo_features.py`
  (`haversine_distance_km`, `add_distance_from_center`), with unit
  tests in `tests/test_geo_features.py`. Uses the identical
  Prinzipalmarkt/Dom reference point (`51.9625, 7.6256`)
  `07_descriptive_analysis.ipynb` used for its distance-from-center
  correlation check, but computed via exact haversine rather than that
  notebook's flat-earth approximation (distances match to well within
  1%, per `test_haversine_distance_km_matches_flat_earth_at_short_range`).
- Trained two otherwise-identical `HistGradientBoostingRegressor`
  variants - `08`'s exact feature set as a control, and the same set
  plus `distance_from_center_km` - on the identical chronological
  8-week test split (106,043 evaluable test rows).
- **Result: the distance feature made zero difference.** Overall MAE
  28.570692 / RMSE 54.529404 for both `gbm_control` and
  `gbm_plus_distance` - identical to six decimal places. The per-station
  table in section 7 shows the same for all 23 stations, including both
  flagged ones (`300037405`: 66.859770 MAE for both variants, still a
  mild regression vs. the 61.549192 baseline; `300038855`: 36.471251 MAE
  for both variants, still a severe regression vs. the 17.332131
  baseline). Section 8 confirms this isn't a rounding coincidence:
  `control_prediction` and `plus_distance_prediction` are exactly equal
  for every test row (`np.array_equal` is `True`), and permutation
  importance for `distance_from_center_km` in the `plus_distance` model
  is effectively zero.
- **Why**: `distance_from_center_km` is a deterministic, many-to-one
  function of `station_id` (one fixed value per station), and
  `station_id` is already a categorical feature in both variants.
  `HistGradientBoostingRegressor`'s categorical splits can already
  partition stations into arbitrary groups across boosting rounds, which
  strictly dominates anything a single derived numeric feature of
  `station_id` could add - so the model never had a reason to split on
  it. This is consistent with `08`'s own permutation-importance finding
  that `station_id` is already the model's 4th-most-important feature;
  distance-from-center was already "priced in" as part of whatever
  `station_id` lets the trees encode.
- **Recommendation: do not adopt `distance_from_center_km` as a GBM/
  LightGBM feature** - it is fully subsumed by the existing `station_id`
  categorical and provably changes nothing, for either station overall
  or the two flagged regression stations specifically. This result does
  *not* close the door on the underlying idea (geography potentially
  explaining some of what `station_id` captures) - it would need a model
  class that can't already freely partition on `station_id` (e.g. a
  linear/spline model, or a model regularizing/sharing information
  *across* stations rather than splitting them apart, such as a
  hierarchical or embedding-based approach) to have a chance of adding
  value. Left as a candidate for that future architecture, not this one.